In [ ]:
import sys
sys.path.append("..")
import utils_for_cluster_first_try as utils
import utils_for_rings_and_discs as gen_and_plot_utils

import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt

import importlib
importlib.reload(utils)


In [ ]:
import pickle
import io
importlib.reload(utils)

from pathlib import Path

DATASET_LOAD_PATH = Path.cwd() / "rings_and_discs_dataset_for_NF.pkl"

with open(DATASET_LOAD_PATH, "rb") as f:
    dataset_dict = pickle.load(f)

In [ ]:
# Normalizing flow
DICT_LOAD_PATH = Path.cwd() / "dict_running_cond_NF.pkl"

class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        else:
            return super().find_class(module, name)

with open(DICT_LOAD_PATH, "rb") as f:
    output_dict = CPU_Unpickler(f).load()


# unsupervised VAE
DICT_LOAD_PATH = Path.cwd() / "dict_running_VAE.pkl"

class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        else:
            return super().find_class(module, name)

with open(DICT_LOAD_PATH, "rb") as f:
    output_dict_VAE = CPU_Unpickler(f).load()

### Plot the results of training

In [ ]:
train_dict = output_dict["train_dict"]
utils.graph_training_output(train_dict,model_name="cond_NF",num_skip_first_epoch=000)

zhats = train_dict["zhats"]


Zhat = output_dict["Zhat"]
model = output_dict["model"]
optimizer = output_dict["optimizer"]
B_est = output_dict["B_est"]
Sigma_Z_est = output_dict["Sigma_Z_est"]
Sigma_X_est = output_dict["Sigma_X_est"]
H_est = output_dict["H_est"]
T_est = output_dict["T_est"]
lambdas_est = output_dict["lambdas_est"]


# we also need the encoder and decoder from the VAE
encoder_mean = output_dict_VAE["encoder_mean"]
decoder_mean = output_dict_VAE["decoder_mean"]

X = dataset_dict["X"]
Y = dataset_dict["Y"]
Z_standard = dataset_dict["Z_standard"]
Z = dataset_dict["Z"]
What = dataset_dict["Zhat"]


U = Zhat@H_est
V = X@T_est

In [ ]:
importlib.reload(utils)

# we want to plot the latent space
colors_1 = Z_standard[:,2]
colors_2 = Z_standard[:,3]
utils.plot_latent_variables(U, colors_1, colors_2, color_1_label="radius_of_hole", color_2_label="width_of_disc", H_est=None)

### In-sample correlation attained

In [ ]:
utils.plot_canonical_variables(U - np.mean(U, axis=0), V - np.mean(V, axis=0), colors_1, colors_2)

In [ ]:
importlib.reload(utils)

#large validation set
DATASET_LOAD_PATH = Path.cwd() / "rings_and_discs_validation_dataset.pkl"

with open(DATASET_LOAD_PATH, "rb") as f:
    dataset_dict_val = pickle.load(f)


Y_val = dataset_dict_val["Y_val"]
X_val = dataset_dict_val["X_val"]
Z_standard_val = dataset_dict_val["Z_standard_val"]
Z_val = dataset_dict_val["Z_val"]
colors_1_val = Z_standard_val[:,2]
colors_2_val = Z_standard_val[:,3]
val_dataset = utils.XYDataset(X_val,Y_val)
val_correlation = utils.out_of_sample_correlation_NF(val_dataset,encoder_mean=encoder_mean,H=output_dict["H_est"],T=output_dict["T_est"],model=output_dict["model"])


print("Out-of-sample canonical correlations:", val_correlation)
print("sum:", np.sum(np.abs(val_correlation)))

What_val = encoder_mean(val_dataset.Y)
Zhat_val = model.inverse(What_val).detach().numpy()
U_val = Zhat_val@H_est
V_val = X_val@T_est

utils.plot_canonical_variables(U_val - np.mean(U_val, axis=0), V_val - np.mean(V_val, axis=0), colors_1_val, colors_2_val)

In [ ]:
utils.out_of_sample_reconstruction_error(val_dataset,encoder_mean,decoder_mean)